# Umbilicus Detection using YOLOv26

This notebook demonstrates a complete workflow for training and evaluating an object detection model for umbilicus localization using the YOLOv26 architecture.

The notebook includes dataset verification, model training, evaluation of training metrics, and inference on validation images.

## Table of Contents

0. Setup & Environment  
1. Import Libraries  
2. Dataset Overview  
3. Dataset Verification  
4. Dataset Visualization  
5. YOLO Dataset Configuration  
6. Model Training  
7. Training Results  
8. Model Inference  
9. Discussion  
10. Conclusion

## 0. Setup & Environment

This section installs the required libraries and prepares the environment for training and running the YOLO object detection model.

In [ ]:
# Install 
%pip install -U ultralytics pillow matplotlib pyyaml

#1.import libraries
import os
import glob
import yaml
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from ultralytics import YOLO

## 2. Dataset Overview

The dataset contains abdominal images with bounding box annotations around the umbilicus.

Images are organized into training(70%), validation(20%), and test(10%) sets following the standard YOLO dataset structure.

## 3. Dataset Verification

Before training the model, we verify that the dataset structure is correct.  
This includes checking directory paths and ensuring that image and label files exist.

In [ ]:
print("Current working directory:")
print(os.getcwd())

print("\nFiles/folders in current directory:")
print(os.listdir())

In [ ]:
# =========================
# USER-DEFINED PATHS
# =========================

DATASET_ROOT = r"D:\Bam\MEng\Coding\Guided system\YOLO training\YOLO8.v3i.yolo26"
DATA_YAML = os.path.join(DATASET_ROOT, "data.yaml")

TRAIN_IMAGES = os.path.join(DATASET_ROOT, "train", "images")
TRAIN_LABELS = os.path.join(DATASET_ROOT, "train", "labels")

VALID_IMAGES = os.path.join(DATASET_ROOT, "valid", "images")
VALID_LABELS = os.path.join(DATASET_ROOT, "valid", "labels")

TEST_IMAGES = os.path.join(DATASET_ROOT, "test", "images")
TEST_LABELS = os.path.join(DATASET_ROOT, "test", "labels")

PROJECT_DIR = r"D:\Bam\MEng\Coding\Guided system\YOLO training\umbilicus_yolo26"
RUN_NAME = "trial1"

MODEL_NAME = "yolo26n.pt"

### Path Check

The following cell checks whether all important dataset paths exist.  
If any path returns `False`, it should be corrected before proceeding.

In [ ]:
paths_to_check = {
    "DATASET_ROOT": DATASET_ROOT,
    "DATA_YAML": DATA_YAML,
    "TRAIN_IMAGES": TRAIN_IMAGES,
    "TRAIN_LABELS": TRAIN_LABELS,
    "VALID_IMAGES": VALID_IMAGES,
    "VALID_LABELS": VALID_LABELS,
    "TEST_IMAGES": TEST_IMAGES,
    "TEST_LABELS": TEST_LABELS,
    "PROJECT_DIR": PROJECT_DIR,
}

for name, path in paths_to_check.items():
    print(f"{name}:")
    print(path)
    print("Exists:", os.path.exists(path))
    print("-" * 60)

In [ ]:
def count_files(folder, extensions=None):
    if not os.path.exists(folder):
        return 0
    if extensions is None:
        return len(os.listdir(folder))
    
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(folder, f"*.{ext}")))
    return len(files)

image_exts = ["jpg", "jpeg", "png"]

print("Train images:", count_files(TRAIN_IMAGES, image_exts))
print("Train labels:", count_files(TRAIN_LABELS, ["txt"]))

print("Valid images:", count_files(VALID_IMAGES, image_exts))
print("Valid labels:", count_files(VALID_LABELS, ["txt"]))

print("Test images:", count_files(TEST_IMAGES, image_exts))
print("Test labels:", count_files(TEST_LABELS, ["txt"]))

## 4. Dataset Visualization

Visualizing sample images helps confirm that the dataset is loaded correctly and provides an overview of the image content before training.

In [ ]:
sample_images = (
    glob.glob(os.path.join(TRAIN_IMAGES, "*.jpg")) +
    glob.glob(os.path.join(TRAIN_IMAGES, "*.jpeg")) +
    glob.glob(os.path.join(TRAIN_IMAGES, "*.png"))
)

print("Number of sample training images found:", len(sample_images))

if len(sample_images) > 0:
    plt.figure(figsize=(12, 8))
    for i, img_path in enumerate(sample_images[:4]):
        img = Image.open(img_path)
        plt.subplot(2, 2, i + 1)
        plt.imshow(img)
        plt.title(os.path.basename(img_path))
        plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No training images found.")

## 5. YOLO Dataset Configuration

The dataset configuration is defined in the `data.yaml` file.  
This file specifies the training and validation paths, as well as the object class used for detection.

In [ ]:
if os.path.exists(DATA_YAML):
    with open(DATA_YAML, "r", encoding="utf-8") as f:
        data_config = yaml.safe_load(f)
    
    print("Contents of data.yaml:")
    print(data_config)
else:
    raise FileNotFoundError(f"data.yaml not found: {DATA_YAML}")

In [ ]:
required_keys = ["train", "val", "names"]
missing_keys = [k for k in required_keys if k not in data_config]

if len(missing_keys) == 0:
    print("data.yaml contains all required keys.")
else:
    print("Missing keys in data.yaml:", missing_keys)

if "names" in data_config:
    print("\nClass names:")
    print(data_config["names"])

## 6. Model Training

In this section, the **YOLOv26 object detection model** is trained using the prepared dataset.

Transfer learning is applied by starting from a pretrained YOLO model and adapting it to the umbilicus detection task.

In [ ]:
# Load pretrained YOLO model
model = YOLO(MODEL_NAME)

# Train the model
results = model.train(
    data=DATA_YAML,
    epochs=20,
    imgsz=640,
    batch=4,
    project=PROJECT_DIR,
    name=RUN_NAME
)

## 7. Training Results

After training, YOLO automatically generates training curves that summarize the model’s learning progress.

These curves include loss values and evaluation metrics such as precision, recall, and mean average precision.

In [ ]:
RUN_DIR = results.save_dir
RESULTS_PNG = os.path.join(RUN_DIR, "results.png")
BEST_MODEL_PATH = os.path.join(RUN_DIR, "weights", "best.pt")
LAST_MODEL_PATH = os.path.join(RUN_DIR, "weights", "last.pt")

print("Run directory:", RUN_DIR)
print("RESULTS_PNG exists:", os.path.exists(RESULTS_PNG))
print("BEST_MODEL_PATH exists:", os.path.exists(BEST_MODEL_PATH))
print("LAST_MODEL_PATH exists:", os.path.exists(LAST_MODEL_PATH))

In [ ]:
if os.path.exists(RESULTS_PNG):
    img = Image.open(RESULTS_PNG)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Training Results")
    plt.show()
else:
    print("results.png not found.")

## 8. Model Inference

The trained model is used to perform inference on validation images.

The output includes bounding boxes showing the predicted location of the umbilicus.

In [ ]:
if os.path.exists(BEST_MODEL_PATH):
    best_model = YOLO(BEST_MODEL_PATH)
    print("Best model loaded successfully.")
else:
    raise FileNotFoundError(f"best.pt not found: {BEST_MODEL_PATH}")

In [ ]:
best_model.predict(
    source=VALID_IMAGES,
    conf=0.25,
    save=True,
    show=True
)

In [ ]:
import glob

predict_folder = os.path.join("runs", "detect", "predict")

images = glob.glob(os.path.join(predict_folder, "*.jpg"))

print("Number of prediction images:", len(images))

plt.figure(figsize=(12,8))

for i, img_path in enumerate(images[:4]):   # show first 4 predictions
    img = Image.open(img_path)
    
    plt.subplot(2,2,i+1)
    plt.imshow(img)
    plt.axis("off")
    
plt.tight_layout()
plt.show()